In [2]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split

from eda_package.data import load_raw_data, clean_data, split_data
from eda_package.features import engineer_features_v2
from eda_package.preprocessor import (
    get_feature_lists,
    create_preprocessor,
    fit_transform_preprocessor,
    transform_preprocessor,
    group_countries
)
from eda_package.registry import COUNTRY_LIMIT, ORDINAL_FEATURES_MAP

In [3]:
from pathlib import Path
import pickle
import pandas as pd

class PreprocessedDataLoader:

    def __init__(self, file_name: str = "preprocessed_data.pkl"):

        #self.path = Path(__file__).resolve().parent.parent / "models" / file_name
        self.path = Path("models") / file_name

        if self.path.exists():
            self.load()
        else:
            self.build()
            self.save()

    def build(self):

        # Load + clean
        df = load_raw_data()
        df = clean_data(df)

        # Feature engineering
        df = group_countries(df, COUNTRY_LIMIT)
        df = engineer_features_v2(df)

        # Split data
        X_train, X_test, y_train, y_test = split_data(df)

        # Preprocessor
        feature_lists = get_feature_lists(X_train, ORDINAL_FEATURES_MAP)
        self.preprocessor = create_preprocessor(feature_lists, ORDINAL_FEATURES_MAP)

        # Transform
        self.X_train_processed = fit_transform_preprocessor(X_train, self.preprocessor)
        self.X_test_processed = transform_preprocessor(X_test, self.preprocessor)

        self.y_train = y_train
        self.y_test = y_test

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)

        with open(self.path, "wb") as f:
            pickle.dump({
                "X_train_processed": self.X_train_processed,
                "X_test_processed": self.X_test_processed,
                "y_train": self.y_train,
                "y_test": self.y_test,
                "preprocessor": self.preprocessor
            }, f)

    def load(self):
        with open(self.path, "rb") as f:
            data = pickle.load(f)

        self.X_train_processed = data["X_train_processed"]
        self.X_test_processed = data["X_test_processed"]
        self.y_train = data["y_train"]
        self.y_test = data["y_test"]
        self.preprocessor = data["preprocessor"]

    def get_data(self):
        return (
            self.X_train_processed,
            self.X_test_processed,
            self.y_train,
            self.y_test,
            self.preprocessor
        )

In [ ]:
# Usage
# data_loader = PreprocessedDataLoader()
# X_train, X_test, y_train, y_test, preprocessor = data_loader.get_data()